In [ ]:
import numpy as np
import casadi as ca
import pandas as pd
import matplotlib.pyplot as plt
import pypesto 
import scipy
# 
from src.kinetics_noor import *
import time
from src.param_estimation import *
import scipy.stats as stats
import seaborn as sns
import time

In [ ]:
# data_import
balanced_met_df   = pd.read_csv('Data/balanced_metabolites.csv', index_col=0)
imbalanced_met_df = pd.read_csv('Data/imbalanced_metabolites.csv', index_col=0)
prot_df           = pd.read_csv('Data/important_proteins.csv', index_col=0)
flux_df           = pd.read_csv('Data/important_fluxes.csv', index_col=0)
cell_needs_df     = pd.read_csv('Data/cellular_needs.csv', index_col=0)

# bounds
imbalanced_bounds = {}
total_min = 1e12
for met in imbalanced_met_df.index:
    
    imbalanced_bounds[met] = (np.where(
        imbalanced_met_df.loc[met].min() > 0, imbalanced_met_df.loc[met].min() * 0.8, 1e-6
    ), imbalanced_met_df.loc[met].max()    * 1.2)
    if imbalanced_bounds[met][0] < total_min:
        total_min = imbalanced_bounds[met][0]
imbalanced_bounds["C_pi"] = (total_min, 10.0)  # Pi is often in the mM range, but can vary widely; set a broad range.



max_max_balanced = balanced_met_df.max().max() * 1.2
min_min_balanced = 1e-6
balanced_bounds = {
    "C_g6p": (min_min_balanced, max_max_balanced), 
    "C_f6p": (min_min_balanced, max_max_balanced), 
    "C_fbp": (min_min_balanced, max_max_balanced), 
    "C_dhap": (min_min_balanced, max_max_balanced), 
    "C_g3p": (min_min_balanced, max_max_balanced), 
    "C_pgp": (min_min_balanced, max_max_balanced), 
    "C_3pg": (min_min_balanced, max_max_balanced), 
    "C_2pg": (min_min_balanced, max_max_balanced), 
    "C_pep": (min_min_balanced, max_max_balanced), 
}

model = EcoliCarbonKinetics(
    bounds_imbalanced_mets=imbalanced_bounds,
    bounds_balanced_mets=balanced_bounds,
)

In [ ]:
import numpy as np

# ── Molar masses [kDa = g/mmol] ───────────────────────────────────────────────
masamolar = {
    'pgi': 59.0,
    'pfk': 36.0,
    'fba': 40.0,
    'tpi': 27.5,
    'gap': 35.0,
    'pgk': 43.7,
    'gpm': 27.0,
    'eno': 46.0
}

def kcat_convert(kcat_s, enzyme):
    MM_g_per_mol = masamolar[enzyme] * 1000   # kDa → g/mol
    
    return kcat_s / MM_g_per_mol * 3600 # convert from per second to per hour (3600 s/h)

# ── Literature parameters ──────────────────────────────────────────────────────
params_from_literature = {
    "v_max_1":    25.739,
    "Ka1_1":      1.0,
    "Ka2_1":      0.01,
    "Ka3_1":      1.0,
    "K_g6p_1":    0.5,
    "kcat_f_2":   None,
    "Ks_g6p_pgi": 1.018,
    "Kp_f6p_pgi": np.mean([0.200, 0.078]),
    "kcat_f_3":   None,
    "Ks_f6p_3":   0.030,
    "Ks_atp_3":   0.060,
    "Kp_fbp_3":   None,
    "Kp_adp_3":   None,
    "kcat_f_4":   kcat_convert(10.33, 'fba'),
    "Ks_fbp_4":   0.240,
    "Kp_g3p_4":   None,
    "Kp_dhap_4":  None,
    "kcat_f_5":   kcat_convert(np.mean([8700, 9000]), 'tpi'),
    "Ks_dhap_5":  1.030,
    "Kp_g3p_5":   None,
    "kcat_f_6":   None,
    "Ks_g3p_6":   0.890,
    "Ks_pi_6":    0.530,
    "Ks_nad_6":   0.045,
    "Kp_pgp_6":   None,
    "Kp_nadh_6":  None,
    "kcat_f_7":   None,
    "Ks_pgp_7":   None,
    "Ks_adp_7":   None,
    "Ks_3pg_7":   None,
    "Ks_atp_7":   None,
    "kcat_f_8":   kcat_convert(220, 'gpm'),
    "Ks_3pg_8":   0.200,
    "Ks_2pg_8":   0.190,
    "kcat_f_9":   None,
    "Ks_2pg_9":   0.100,
    "Ks_pep_9":   None,
}

# ── Distribution hyperparameters ───────────────────────────────────────────────
median_kcat_central_met         = 79
rang                            = [50, 90]

median_kcat_over_kM_central_met = 410_000 / 1000    # mM-1s-1 = 410
median_kcat_over_kM_range       = [300_000/1000, 500_000/1000]

median_Km  = 0.500          # mM  
Km_range   = [0.010, 0.700] # mM  

params_to_set     = ['kcat_f_2', 'kcat_f_3', 'kcat_f_6', 'kcat_f_7', 'kcat_f_9']
params_kcat_fixed = ['kcat_f_4', 'kcat_f_5']   # kcat known, only Kms missing

# ── Sampler ────────────────────────────────────────────────────────────────────
def sample_missing_params(params, median_kcat, kcat_range, median_kcat_kM,
                          kcat_kM_range, median_Km, Km_range, seed=None):
    if seed is not None:
        np.random.seed(seed)

    def sample_lognormal(median, lo, hi):
        mu    = np.log(median)
        sigma = (np.log(hi) - np.log(lo)) / (2 * 1.96)
        return np.exp(np.random.normal(mu, sigma))

    suffix_to_enzyme = {
        '2': 'pgi', '3': 'pfk', '4': 'fba', '5': 'tpi',
        '6': 'gap', '7': 'pgk', '8': 'gpm', '9': 'eno'
    }

    missing_kms = {
        'pgi': [],
        'pfk': ['Kp_fbp_3', 'Kp_adp_3'],
        'fba': ['Kp_g3p_4', 'Kp_dhap_4'],
        'tpi': ['Kp_g3p_5'],
        'gap': ['Kp_pgp_6', 'Kp_nadh_6'],
        'pgk': ['Ks_pgp_7', 'Ks_adp_7', 'Ks_3pg_7', 'Ks_atp_7'],
        'gpm': [],
        'eno': ['Ks_pep_9'],
    }

    sampled = params.copy()

    # sample kcat + Kms via kcat/Km ratio
    for kcat_key in params_to_set:
        enzyme            = suffix_to_enzyme[kcat_key.split('_')[-1]]
        kcat_s            = sample_lognormal(median_kcat, *kcat_range)
        sampled[kcat_key] = kcat_convert(kcat_s, enzyme)
        for km_key in missing_kms[enzyme]:
            kcat_kM         = sample_lognormal(median_kcat_kM, *kcat_kM_range)
            sampled[km_key] = kcat_s / kcat_kM                              # mM

    # fixed-kcat enzymes: sample Km directly from Fig 1C distribution
    for kcat_key in params_kcat_fixed:
        enzyme = suffix_to_enzyme[kcat_key.split('_')[-1]]
        for km_key in missing_kms[enzyme]:
            sampled[km_key] = sample_lognormal(median_Km, *Km_range)       # mM

    return sampled

sampled_params = sample_missing_params(
    params          = params_from_literature,
    median_kcat     = median_kcat_central_met,
    kcat_range      = rang,
    median_kcat_kM  = median_kcat_over_kM_central_met,
    kcat_kM_range   = median_kcat_over_kM_range,
    median_Km       = median_Km,
    Km_range        = Km_range,
    seed            = 42
)

param_latex = {
    "v_max_1":    r"$v^{\max}_{PTS}$",
    "Ka1_1":      r"$K_{a1}$",
    "Ka2_1":      r"$K_{a2}$",
    "Ka3_1":      r"$K_{a3}$",
    "K_g6p_1":    r"$K_{g6p}$",
    "Ks_g6p_pgi": r"$K_{s,g6p}^{2}$",
    "Kp_f6p_pgi": r"$K_{p,f6p}^{2}$",
    "kcat_f_2":   r"$k^{+}_{cat,2}$",
    "Ks_f6p_3":   r"$K_{s,f6p}^{3}$",
    "Ks_atp_3":   r"$K_{s,atp}^{3}$",
    "Kp_fbp_3":   r"$K_{p,fbp}^{3}$",
    "Kp_adp_3":   r"$K_{p,adp}^{3}$",
    "kcat_f_3":   r"$k^{+}_{cat,3}$",
    "Ks_fbp_4":   r"$K_{s,fbp}^{4}$",
    "Kp_g3p_4":   r"$K_{p,g3p}^{4}$",
    "Kp_dhap_4":  r"$K_{p,dhap}^{4}$",
    "kcat_f_4":   r"$k^{+}_{cat,4}$",
    "kcat_f_5":   r"$k^{+}_{cat,5}$",
    "Ks_dhap_5":  r"$K_{s,dhap}^{5}$",
    "Kp_g3p_5":   r"$K_{p,g3p}^{5}$",
    "kcat_f_6":   r"$k^{+}_{cat,6}$",
    "Ks_g3p_6":   r"$K_{s,g3p}^{6}$",
    "Ks_pi_6":    r"$K_{s,pi}^{6}$",
    "Ks_nad_6":   r"$K_{s,nad}^{6}$",
    "Kp_pgp_6":   r"$K_{p,pgp}^{6}$",
    "Kp_nadh_6":  r"$K_{p,nadh}^{6}$",
    "kcat_f_7":   r"$k^{+}_{cat,7}$",
    "Ks_pgp_7":   r"$K_{s,pgp}^{7}$",
    "Ks_adp_7":   r"$K_{s,adp}^{7}$",
    "Ks_3pg_7":   r"$K_{p,3pg}^{7}$",
    "Ks_atp_7":   r"$K_{p,atp}^{7}$",
    "kcat_f_8":   r"$k^{+}_{cat,8}$",
    "Ks_3pg_8":   r"$K_{s,3pg}^{8}$",
    "Ks_2pg_8":   r"$K_{p,2pg}^{8}$",
    "kcat_f_9":   r"$k^{+}_{cat,9}$",
    "Ks_2pg_9":   r"$K_{s,2pg}^{9}$",
    "Ks_pep_9":   r"$K_{p,pep}^{9}$",
}

df_params = pd.DataFrame([
    {"Parameter": param_latex[k], "Value": float(sampled_params[k]),
     "Source": "Literature" if params_from_literature.get(k) is not None else "Sampled"}
    for k in model.params_keys
])

In [ ]:
from src.sentitivity import compute_sensitivity

# Load measurement uncertainty (Q) -- shared across all conditions
Q_bal  = pd.read_csv("Data/balanced_metabolites_Q.csv", index_col=0).squeeze()
Q_flux = pd.read_csv("Data/important_fluxes_Q.csv",     index_col=0).squeeze()

# Build measurement_error dict: key -> SD, np.nan if output was not measured
measurement_error = {k: Q_bal.get(k, np.nan)  for k in model.balanced_keys}
measurement_error.update({k: Q_flux.get(k, np.nan) for k in model.flux_keys})

measured_keys = [k for k in measurement_error if np.isfinite(measurement_error[k])]
print(f"Measured outputs: {len(measured_keys)} / {len(model.balanced_keys) + len(model.flux_keys)}")
print(measured_keys)

In [ ]:
# Load and transpose so that conditions are rows and variables are columns
prot_df       = pd.read_csv("Data/important_proteins.csv", index_col=0)
cell_needs_df = pd.read_csv("Data/cellular_needs.csv",     index_col=0)

input_enzyme     = prot_df.T         # (n_conditions, n_enzymes)
input_cell_needs = cell_needs_df.T   # (n_conditions, n_balanced_mets)

# Drop conditions where any enzyme measurement is missing
input_enzyme = input_enzyme.dropna()

# Intersect with cell_needs to get fully valid conditions
valid_conds = input_enzyme.index.intersection(input_cell_needs.index)
input_enzyme     = input_enzyme.loc[valid_conds]
input_cell_needs = input_cell_needs.loc[valid_conds]

conditions = [{"condition": c} for c in valid_conds]
print(f"Valid conditions ({len(conditions)}): {list(valid_conds)}")

In [ ]:


t0 = time.time()
G_total, corr, diag = compute_sensitivity(
    model             = model,
    params_estimate   = sampled_params,
    input_enzyme      = input_enzyme,
    input_cell_needs  = input_cell_needs,
    conditions        = conditions,
    measurement_error = measurement_error,
)


In [ ]:
# Order-of-magnitude check: log10(predicted / measured) for 3 conditions.
# Values inside the grey band (-1, +1) are within one order of magnitude -- acceptable.
sample_keys = [valid_conds[0], valid_conds[len(valid_conds) // 2], valid_conds[-1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, cond_key in zip(axes, sample_keys):
    enzymes_c    = input_enzyme.loc[cond_key].to_dict()
    cell_needs_c = input_cell_needs.loc[cond_key].to_dict()
    df_pred, ss_res = model.solve_steady_state(
        enzymes_c, sampled_params, cell_needs_c, condition_key=cond_key
    )
    pred = df_pred.iloc[0].to_dict()

    # collect outputs where both measured and predicted are positive
    labels, log_ratios, colors = [], [], []
    for k in model.balanced_keys + model.flux_keys:
        src = balanced_met_df if k.startswith('C_') else flux_df
        if k not in src.index or cond_key not in src.columns:
            continue
        v_meas = src.loc[k, cond_key]
        v_pred = pred.get(k)
        if pd.isna(v_meas) or v_meas <= 0 or v_pred is None or float(v_pred) <= 0:
            continue
        ratio = np.log10(float(v_pred) / float(v_meas))
        labels.append(k.replace('C_', '').replace('v_', ''))
        log_ratios.append(ratio)
        colors.append('steelblue' if k.startswith('C_') else 'tomato')

    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, log_ratios, color=colors, alpha=0.75, edgecolor='k', linewidth=0.4)

    # highlight bars outside ±1 order of magnitude
    for bar, ratio in zip(bars, log_ratios):
        if abs(ratio) > 1:
            bar.set_edgecolor('black')
            bar.set_linewidth(1.5)
            bar.set_hatch('//')

    ax.axvspan(-1, 1, color='grey', alpha=0.12, label='±1 order of magnitude')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('log$_{10}$(predicted / measured)', fontsize=9)
    ax.set_title(f'{cond_key}   (||Sv-b|| = {ss_res:.1e})', fontsize=9)
    ax.legend(fontsize=7)

from matplotlib.patches import Patch
fig.legend(
    handles=[Patch(color='steelblue', label='concentration (mM)'),
             Patch(color='tomato',    label='flux (mmol/gDCW/h)'),
             Patch(facecolor='grey',  alpha=0.3, label='within 1 order of magnitude')],
    loc='lower center', ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.08)
)
plt.suptitle('Order-of-magnitude check: log$_{10}$(pred / meas) per output', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Mean absolute relative sensitivity -- split by concentrations and fluxes so each
# has its own color scale (flux sensitivities are structurally suppressed by S*dv/dtheta=0
# and visually swamped when concentrations with tiny steady-state values inflate 1/y).
G_mean = np.mean([np.abs(G) for G in diag['G_list']], axis=0)

# split measured outputs into concentrations and fluxes
meas_bal  = [k for k in diag['measured'] if k.startswith('C_')]
meas_flux = [k for k in diag['measured'] if k.startswith('v_')]
n_meas_bal  = len(meas_bal)
n_meas_flux = len(meas_flux)

top_n   = 15
top_idx = np.argsort(G_mean.max(axis=0))[::-1][:top_n]
xlabels = [model.params_keys[i] for i in top_idx]

fig, (ax_c, ax_f) = plt.subplots(2, 1, figsize=(14, 7),
                                  gridspec_kw={'height_ratios': [n_meas_bal, n_meas_flux],
                                               'hspace': 0.05})

# concentrations
im_c = ax_c.imshow(G_mean[:n_meas_bal, :][:, top_idx], aspect='auto', cmap='viridis')
ax_c.set_xticks([])
ax_c.set_yticks(range(n_meas_bal))
ax_c.set_yticklabels(meas_bal, fontsize=8)
ax_c.set_title("Mean |relative sensitivity| -- top 15 parameters", fontsize=10)
plt.colorbar(im_c, ax=ax_c, label="|d ln C / d ln θ|")

# fluxes
im_f = ax_f.imshow(G_mean[n_meas_bal:, :][:, top_idx], aspect='auto', cmap='viridis')
ax_f.set_xticks(range(top_n))
ax_f.set_xticklabels(xlabels, rotation=55, ha='right', fontsize=8)
ax_f.set_yticks(range(n_meas_flux))
ax_f.set_yticklabels(meas_flux, fontsize=8)
plt.colorbar(im_f, ax=ax_f, label="|d ln v / d ln θ|")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle as MplRect

# `corr` (n x n correlation matrix) must already exist in scope.

reaction_groups = [
    ("PTS",  0,  4,  '#d73027'), ("PGI",  5,  7,  '#4575b4'),
    ("PFK",  8, 12,  '#1a9850'), ("FBA", 13, 16,  '#762a83'),
    ("TPI", 17, 19,  '#e08214'), ("GAP", 20, 25,  '#8c510a'),
    ("PGK", 26, 30,  '#de77ae'), ("GPM", 31, 33,  '#80cdc1'),
    ("ENO", 34, 36,  '#74add1'),
]

# LaTeX labels in model.params_keys order (37 entries)
param_labels = [
    r"$v^{\max}_{PTS}$", r"$K_{a1}$", r"$K_{a2}$", r"$K_{a3}$", r"$K_{g6p}$",
    r"$K_{s,g6p}^{2}$", r"$K_{p,f6p}^{2}$", r"$k^{+}_{cat,2}$",
    r"$K_{s,f6p}^{3}$", r"$K_{s,atp}^{3}$", r"$K_{p,fbp}^{3}$", r"$K_{p,adp}^{3}$", r"$k^{+}_{cat,3}$",
    r"$K_{s,fbp}^{4}$", r"$K_{p,g3p}^{4}$", r"$K_{p,dhap}^{4}$", r"$k^{+}_{cat,4}$",
    r"$k^{+}_{cat,5}$", r"$K_{s,dhap}^{5}$", r"$K_{p,g3p}^{5}$",
    r"$k^{+}_{cat,6}$", r"$K_{s,g3p}^{6}$", r"$K_{s,pi}^{6}$", r"$K_{s,nad}^{6}$", r"$K_{p,pgp}^{6}$", r"$K_{p,nadh}^{6}$",
    r"$k^{+}_{cat,7}$", r"$K_{s,pgp}^{7}$", r"$K_{s,adp}^{7}$", r"$K_{p,3pg}^{7}$", r"$K_{p,atp}^{7}$",
    r"$k^{+}_{cat,8}$", r"$K_{s,3pg}^{8}$", r"$K_{p,2pg}^{8}$",
    r"$k^{+}_{cat,9}$", r"$K_{s,2pg}^{9}$", r"$K_{p,pep}^{9}$",
]

# ── reorder: within each reaction, k_cat first then K_m (PTS unchanged) ─────
# Applied to BOTH the labels and the matrix so they stay consistent.
_KCAT0 = {7, 12, 16, 17, 20, 26, 31, 34}
_PTS0  = {0, 1, 2, 3, 4}
def _t0(i): return 'pts' if i in _PTS0 else ('kcat' if i in _KCAT0 else 'km')
_rank = {'kcat': 0, 'km': 1, 'pts': 0}
_perm = []
for _nm, _is, _ie, _c in reaction_groups:
    _perm += sorted(range(_is, _ie + 1), key=lambda i: (_rank[_t0(i)], i))
param_labels = [param_labels[i] for i in _perm]
corr = corr[np.ix_(_perm, _perm)]
KCAT_IDX = {p for p, o in enumerate(_perm) if o in _KCAT0}   # positions in NEW order
PTS_IDX  = {p for p, o in enumerate(_perm) if o in _PTS0}

THRESH = 0.0
corr_masked = np.where(np.abs(corr) >= THRESH, corr, np.nan)
np.fill_diagonal(corr_masked, np.nan)
n = len(param_labels)

mpl.rcParams.update({
    'font.family':      'sans-serif',
    'font.size':         9,
    'axes.linewidth':    0.6,
    'mathtext.fontset':  'dejavusans',
})

# Diverging map; masked (sub-threshold) cells render neutral gray so the empty
# grid stays visible against a white page and the strong cells stand out.
cmap = mpl.colormaps['RdBu_r'].copy()
cmap.set_bad('#e9e9ec')

# ── sizes tuned for LaTeX legibility ───────────────────────────────────────
GAP      = 0.35   # gap between matrix and reaction strip (cells)
STRIP    = 1.4    # reaction-strip thickness (cells)  [eased so right-strip text fits]
TICK_FS  = 11     # parameter tick-label size         [larger labels]
STRIP_FS = 9      # reaction-strip text size

fig, ax = plt.subplots(figsize=(10.5, 10.5), dpi=200)
im = ax.imshow(corr_masked, cmap=cmap, vmin=-1, vmax=1,
               interpolation='none', aspect='equal')

# expand limits so the strips live inside the same coordinate system
ax.set_xlim(-0.5, n - 0.5 + GAP + STRIP)
ax.set_ylim(n - 0.5, -0.5 - GAP - STRIP)   # (bottom, top); y inverted

ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(param_labels, rotation=90, fontsize=TICK_FS, ha='center')
ax.set_yticklabels(param_labels, fontsize=TICK_FS)

# parameter TYPE coloring (dark, colorblind-safe, high contrast on white)
# KCAT_IDX / PTS_IDX were computed above for the reordered axis.
TYPE_COLOR = {'kcat': '#117733', 'km': '#332288', 'pts': '#882255'}
def _type_color(i):
    if i in PTS_IDX:  return TYPE_COLOR['pts']
    if i in KCAT_IDX: return TYPE_COLOR['kcat']
    return TYPE_COLOR['km']                        # everything else = K_s / K_p (Km-type)
for i, t in enumerate(ax.get_xticklabels()):
    t.set_color(_type_color(i)); t.set_fontweight('semibold')
for i, t in enumerate(ax.get_yticklabels()):
    t.set_color(_type_color(i)); t.set_fontweight('semibold')
ax.tick_params(length=2, width=0.5, pad=2)

# faint white cell gridlines, confined to the matrix square
for k in np.arange(-0.5, n, 1):
    ax.plot([-0.5, n - 0.5], [k, k], color='white', lw=0.5, zorder=1)
    ax.plot([k, k], [-0.5, n - 0.5], color='white', lw=0.5, zorder=1)

# heavy block separators + colored on-diagonal frames (color <-> position)
for name, i_s, i_e, col in reaction_groups:
    if i_s != 0:
        ax.plot([i_s - 0.5, i_s - 0.5], [-0.5, n - 0.5], color='#2b2b2b', lw=1.1, zorder=3)
        ax.plot([-0.5, n - 0.5], [i_s - 0.5, i_s - 0.5], color='#2b2b2b', lw=1.1, zorder=3)
    ax.add_patch(MplRect((i_s - 0.5, i_s - 0.5), i_e - i_s + 1, i_e - i_s + 1,
                         fill=False, edgecolor=col, lw=1.6, zorder=5))
ax.add_patch(MplRect((-0.5, -0.5), n, n, fill=False, edgecolor='#2b2b2b', lw=1.1, zorder=6))

# reaction strips (top + right), in-axes -> always aligned
top_y   = -0.5 - GAP - STRIP
right_x = n - 0.5 + GAP
for name, i_s, i_e, col in reaction_groups:
    w = i_e - i_s + 1
    ax.add_patch(MplRect((i_s - 0.5, top_y), w, STRIP,
                         facecolor=col, edgecolor='white', lw=1.0, zorder=4))
    ax.text((i_s + i_e) / 2, top_y + STRIP / 2, name, ha='center', va='center',
            fontsize=STRIP_FS, fontweight='bold', color='white', zorder=5)
    ax.add_patch(MplRect((right_x, i_s - 0.5), STRIP, w,
                         facecolor=col, edgecolor='white', lw=1.0, zorder=4))
    ax.text(right_x + STRIP / 2, (i_s + i_e) / 2, name, ha='center', va='center',
            fontsize=STRIP_FS, fontweight='bold', color='white', rotation=270, zorder=5)

for s in ('top', 'right', 'bottom', 'left'):
    ax.spines[s].set_visible(False)

# size-matched colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03, shrink=0.6)
cbar.set_label(r'Correlation  $r_{ij}$', fontsize=11, labelpad=4)
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar.ax.tick_params(labelsize=9, length=3, width=0.5)
cbar.outline.set_linewidth(0.6)

fig.savefig('figures/corr_not_masked.pdf', dpi=300, bbox_inches='tight')
plt.show()

mpl.rcParams.update(mpl.rcParamsDefault)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle as MplRect

# `corr` (n x n correlation matrix) must already exist in scope.

reaction_groups = [
    ("PTS",  0,  4,  '#d73027'), ("PGI",  5,  7,  '#4575b4'),
    ("PFK",  8, 12,  '#1a9850'), ("FBA", 13, 16,  '#762a83'),
    ("TPI", 17, 19,  '#e08214'), ("GAP", 20, 25,  '#8c510a'),
    ("PGK", 26, 30,  '#de77ae'), ("GPM", 31, 33,  '#80cdc1'),
    ("ENO", 34, 36,  '#74add1'),
]

# LaTeX labels in model.params_keys order (37 entries)
param_labels = [
    r"$v^{\max}_{PTS}$", r"$K_{a1}$", r"$K_{a2}$", r"$K_{a3}$", r"$K_{g6p}$",
    r"$K_{s,g6p}^{2}$", r"$K_{p,f6p}^{2}$", r"$k^{+}_{cat,2}$",
    r"$K_{s,f6p}^{3}$", r"$K_{s,atp}^{3}$", r"$K_{p,fbp}^{3}$", r"$K_{p,adp}^{3}$", r"$k^{+}_{cat,3}$",
    r"$K_{s,fbp}^{4}$", r"$K_{p,g3p}^{4}$", r"$K_{p,dhap}^{4}$", r"$k^{+}_{cat,4}$",
    r"$k^{+}_{cat,5}$", r"$K_{s,dhap}^{5}$", r"$K_{p,g3p}^{5}$",
    r"$k^{+}_{cat,6}$", r"$K_{s,g3p}^{6}$", r"$K_{s,pi}^{6}$", r"$K_{s,nad}^{6}$", r"$K_{p,pgp}^{6}$", r"$K_{p,nadh}^{6}$",
    r"$k^{+}_{cat,7}$", r"$K_{s,pgp}^{7}$", r"$K_{s,adp}^{7}$", r"$K_{p,3pg}^{7}$", r"$K_{p,atp}^{7}$",
    r"$k^{+}_{cat,8}$", r"$K_{s,3pg}^{8}$", r"$K_{p,2pg}^{8}$",
    r"$k^{+}_{cat,9}$", r"$K_{s,2pg}^{9}$", r"$K_{p,pep}^{9}$",
]

# ── reorder: within each reaction, k_cat first then K_m (PTS unchanged) ─────
# Applied to BOTH the labels and the matrix so they stay consistent.
_KCAT0 = {7, 12, 16, 17, 20, 26, 31, 34}
_PTS0  = {0, 1, 2, 3, 4}
def _t0(i): return 'pts' if i in _PTS0 else ('kcat' if i in _KCAT0 else 'km')
_rank = {'kcat': 0, 'km': 1, 'pts': 0}
_perm = []
for _nm, _is, _ie, _c in reaction_groups:
    _perm += sorted(range(_is, _ie + 1), key=lambda i: (_rank[_t0(i)], i))
param_labels = [param_labels[i] for i in _perm]
corr = corr[np.ix_(_perm, _perm)]
KCAT_IDX = {p for p, o in enumerate(_perm) if o in _KCAT0}   # positions in NEW order
PTS_IDX  = {p for p, o in enumerate(_perm) if o in _PTS0}

THRESH = 0.9
corr_masked = np.where(np.abs(corr) >= THRESH, corr, np.nan)
np.fill_diagonal(corr_masked, np.nan)
n = len(param_labels)

mpl.rcParams.update({
    'font.family':      'sans-serif',
    'font.size':         9,
    'axes.linewidth':    0.6,
    'mathtext.fontset':  'dejavusans',
})

# Diverging map; masked (sub-threshold) cells render neutral gray so the empty
# grid stays visible against a white page and the strong cells stand out.
cmap = mpl.colormaps['RdBu_r'].copy()
cmap.set_bad('#e9e9ec')

# ── sizes tuned for LaTeX legibility ───────────────────────────────────────
GAP      = 0.35   # gap between matrix and reaction strip (cells)
STRIP    = 1.4    # reaction-strip thickness (cells)  [eased so right-strip text fits]
TICK_FS  = 11     # parameter tick-label size         [larger labels]
STRIP_FS = 9      # reaction-strip text size

fig, ax = plt.subplots(figsize=(10.5, 10.5), dpi=200)
im = ax.imshow(corr_masked, cmap=cmap, vmin=-1, vmax=1,
               interpolation='none', aspect='equal')

# expand limits so the strips live inside the same coordinate system
ax.set_xlim(-0.5, n - 0.5 + GAP + STRIP)
ax.set_ylim(n - 0.5, -0.5 - GAP - STRIP)   # (bottom, top); y inverted

ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(param_labels, rotation=90, fontsize=TICK_FS, ha='center')
ax.set_yticklabels(param_labels, fontsize=TICK_FS)

# parameter TYPE coloring (dark, colorblind-safe, high contrast on white)
# KCAT_IDX / PTS_IDX were computed above for the reordered axis.
TYPE_COLOR = {'kcat': '#117733', 'km': '#332288', 'pts': '#882255'}
def _type_color(i):
    if i in PTS_IDX:  return TYPE_COLOR['pts']
    if i in KCAT_IDX: return TYPE_COLOR['kcat']
    return TYPE_COLOR['km']                        # everything else = K_s / K_p (Km-type)
for i, t in enumerate(ax.get_xticklabels()):
    t.set_color(_type_color(i)); t.set_fontweight('semibold')
for i, t in enumerate(ax.get_yticklabels()):
    t.set_color(_type_color(i)); t.set_fontweight('semibold')
ax.tick_params(length=2, width=0.5, pad=2)

# faint white cell gridlines, confined to the matrix square
for k in np.arange(-0.5, n, 1):
    ax.plot([-0.5, n - 0.5], [k, k], color='white', lw=0.5, zorder=1)
    ax.plot([k, k], [-0.5, n - 0.5], color='white', lw=0.5, zorder=1)

# heavy block separators + colored on-diagonal frames (color <-> position)
for name, i_s, i_e, col in reaction_groups:
    if i_s != 0:
        ax.plot([i_s - 0.5, i_s - 0.5], [-0.5, n - 0.5], color='#2b2b2b', lw=1.1, zorder=3)
        ax.plot([-0.5, n - 0.5], [i_s - 0.5, i_s - 0.5], color='#2b2b2b', lw=1.1, zorder=3)
    ax.add_patch(MplRect((i_s - 0.5, i_s - 0.5), i_e - i_s + 1, i_e - i_s + 1,
                         fill=False, edgecolor=col, lw=1.6, zorder=5))
ax.add_patch(MplRect((-0.5, -0.5), n, n, fill=False, edgecolor='#2b2b2b', lw=1.1, zorder=6))

# reaction strips (top + right), in-axes -> always aligned
top_y   = -0.5 - GAP - STRIP
right_x = n - 0.5 + GAP
for name, i_s, i_e, col in reaction_groups:
    w = i_e - i_s + 1
    ax.add_patch(MplRect((i_s - 0.5, top_y), w, STRIP,
                         facecolor=col, edgecolor='white', lw=1.0, zorder=4))
    ax.text((i_s + i_e) / 2, top_y + STRIP / 2, name, ha='center', va='center',
            fontsize=STRIP_FS, fontweight='bold', color='white', zorder=5)
    ax.add_patch(MplRect((right_x, i_s - 0.5), STRIP, w,
                         facecolor=col, edgecolor='white', lw=1.0, zorder=4))
    ax.text(right_x + STRIP / 2, (i_s + i_e) / 2, name, ha='center', va='center',
            fontsize=STRIP_FS, fontweight='bold', color='white', rotation=270, zorder=5)

for s in ('top', 'right', 'bottom', 'left'):
    ax.spines[s].set_visible(False)

# size-matched colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03, shrink=0.6)
cbar.set_label(r'Correlation  $r_{ij}$', fontsize=11, labelpad=4)
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar.ax.tick_params(labelsize=9, length=3, width=0.5)
cbar.outline.set_linewidth(0.6)

fig.savefig('figures/corr_masked.pdf', dpi=300, bbox_inches='tight')
plt.show()

mpl.rcParams.update(mpl.rcParamsDefault)

In [ ]:

# dict with correlated parameters 
df_corr = pd.DataFrame(corr, index=model.params_keys, columns=model.params_keys)
list_corrs = []
threshold = 0.90
for i in range(len(df_corr)):
    param_i = df_corr.index[i]
    for j in range(i, len(df_corr)):
        param_j = df_corr.columns[j]
        if i != j and abs(df_corr.iloc[i, j]) >= threshold:
            list_corrs.append((param_i, param_j, df_corr.iloc[i, j]))
            
# Print correlated parameters

df_corrs = pd.DataFrame(list_corrs, columns=['Parameter 1', 'Parameter 2', 'Correlation'])
print(df_corrs.to_latex(index=False))


In [ ]:
from joblib import Parallel, delayed
from scipy.stats import spearmanr

N_ITER_OAT = 15

def perturb_param_oat(param, constants, enzymes_df, needs_df, model_kwargs, n_iter=N_ITER_OAT):
    m        = EcoliCarbonKinetics(**model_kwargs)
    out_keys = m.balanced_keys + m.flux_keys
    nom      = constants[param]

    for col in enzymes_df.columns:
        try:
            m.solve_steady_state(enzymes_df[col].to_dict(), constants,
                                 needs_df[col].to_dict(), condition_key=col)
        except Exception:
            pass

    param_values, avg_ys = [], []
    for _ in range(n_iter):
        value = np.random.normal(nom, nom * 0.5)
        while value <= nom * 0.01:
            value = np.random.normal(nom, nom * 0.5)
        c = {**constants, param: value}

        accum, n_ok = np.zeros(len(out_keys)), 0
        for col in enzymes_df.columns:
            try:
                df_i, _ = m.solve_steady_state(enzymes_df[col].to_dict(), c,
                                               needs_df[col].to_dict(), condition_key=col)
                accum += df_i.iloc[0][out_keys].values.astype(float)
                n_ok  += 1
            except Exception:
                pass
        if n_ok > 0:
            param_values.append(value)
            avg_ys.append(accum / n_ok)

    if len(param_values) < 4:
        return param, np.full(len(out_keys), np.nan)

    Y   = np.array(avg_ys)
    X   = np.array(param_values)
    rho = np.full(len(out_keys), np.nan)
    for i in range(len(out_keys)):
        mask = np.isfinite(Y[:, i])
        if mask.sum() > 3:
            rho[i], _ = spearmanr(X[mask], Y[mask, i])
    return param, rho

t0 = time.time()
oat_results = Parallel(n_jobs=-1, backend='loky')(
    delayed(perturb_param_oat)(
        param, sampled_params,
        input_enzyme.T, input_cell_needs.T,
        model_kwargs={'bounds_imbalanced_mets': imbalanced_bounds,
                      'bounds_balanced_mets':   balanced_bounds},
    )
    for param in model.params_keys
)
print(f'OAT sensitivity done in {time.time()-t0:.1f} s')

out_keys_oat = model.balanced_keys + model.flux_keys
param_to_idx = {p: i for i, p in enumerate(model.params_keys)}
rho_oat      = np.full((len(out_keys_oat), len(model.params_keys)), np.nan)
for param, rho in oat_results:
    rho_oat[:, param_to_idx[param]] = rho
print('rho_oat shape:', rho_oat.shape)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle as MplRect
from matplotlib.transforms import blended_transform_factory

reaction_groups = [
    ('PTS',  0,  4,  '#d73027'), ('PGI',  5,  7,  '#4575b4'),
    ('PFK',  8, 12,  '#1a9850'), ('FBA', 13, 16,  '#762a83'),
    ('TPI', 17, 19,  '#e08214'), ('GAP', 20, 25,  '#8c510a'),
    ('PGK', 26, 30,  '#de77ae'), ('GPM', 31, 33,  '#80cdc1'),
    ('ENO', 34, 36,  '#74add1'),
]

param_labels = [param_latex[k] for k in model.params_keys]

bal_keys  = model.balanced_keys
flux_keys = model.flux_keys
bal_idx   = [out_keys_oat.index(k) for k in bal_keys]
flux_idx  = [out_keys_oat.index(k) for k in flux_keys]
row_idx   = bal_idx + flux_idx
row_labels_oat = [k.split('_', 1)[1] for k in bal_keys + flux_keys]
bal_count = len(bal_idx)
n_out, n_par = len(row_idx), len(model.params_keys)

# reorder: kcat first within each reaction group (same as MC plot)
_KCAT0 = {7, 12, 16, 17, 20, 26, 31, 34}
_PTS0  = {0, 1, 2, 3, 4}
def _t0(i): return 'pts' if i in _PTS0 else ('kcat' if i in _KCAT0 else 'km')
_rank = {'kcat': 0, 'km': 1, 'pts': 0}
_perm = []
for _nm, _is, _ie, _c in reaction_groups:
    _perm += sorted(range(_is, _ie + 1), key=lambda i: (_rank[_t0(i)], i))
param_labels = [param_labels[i] for i in _perm]
rho_oat_plot = rho_oat[:, _perm]
KCAT_IDX = {p for p, o in enumerate(_perm) if o in _KCAT0}
PTS_IDX  = {p for p, o in enumerate(_perm) if o in _PTS0}

THRESH = 0.75
rho_masked = np.where(np.abs(rho_oat_plot[row_idx, :]) >= THRESH, rho_oat_plot[row_idx, :], np.nan)

mpl.rcParams.update({'font.family': 'sans-serif', 'font.size': 9,
                     'axes.linewidth': 0.6, 'mathtext.fontset': 'dejavusans'})

cmap = mpl.colormaps['RdBu_r'].copy()
cmap.set_bad('#e9e9ec')

TYPE_COLOR = {'kcat': '#117733', 'km': '#332288', 'pts': '#882255'}
def _tc(i):
    if i in PTS_IDX:  return TYPE_COLOR['pts']
    if i in KCAT_IDX: return TYPE_COLOR['kcat']
    return TYPE_COLOR['km']

GAP_R, STRIP_R = 0.3, 1.0
TOP_STRIP_Y, TOP_STRIP_H = 1.004, 0.026

fig, ax = plt.subplots(figsize=(13.5, 8.5), dpi=200)
im = ax.imshow(rho_masked, aspect='auto', cmap=cmap,
               vmin=-1, vmax=1, interpolation='none')

ax.set_xticks(range(n_par))
ax.set_xticklabels(param_labels, rotation=90, fontsize=10, ha='center')
for i, t in enumerate(ax.get_xticklabels()):
    t.set_color(_tc(i)); t.set_fontweight('semibold')
ax.set_yticks(range(n_out))
ax.set_yticklabels(row_labels_oat, fontsize=9)
ax.tick_params(length=2, width=0.5, pad=2)

for k in np.arange(-0.5, n_par, 1):
    ax.plot([k, k], [-0.5, n_out - 0.5], color='white', lw=0.4, zorder=1)
for k in np.arange(-0.5, n_out, 1):
    ax.plot([-0.5, n_par - 0.5], [k, k], color='white', lw=0.4, zorder=1)

for name, i_s, i_e, col in reaction_groups:
    if i_s != 0:
        ax.plot([i_s - 0.5, i_s - 0.5], [-0.5, n_out - 0.5],
                color='#2b2b2b', lw=1.0, zorder=3)

ax.set_xlim(-0.5, n_par - 0.5 + GAP_R + STRIP_R)
band_x = n_par - 0.5 + GAP_R
ax.axhline(bal_count - 0.5, color='#2b2b2b', lw=1.5, zorder=3)
for lab, y0, y1, col in [('concentration', -0.5, bal_count - 0.5, '#0F7173'),
                          ('flux',  bal_count - 0.5, n_out - 0.5, '#7B2D8E')]:
    ax.add_patch(MplRect((band_x, y0), STRIP_R, y1 - y0,
                         facecolor=col, edgecolor='white', lw=0.8,
                         clip_on=False, zorder=4))
    ax.text(band_x + STRIP_R / 2, (y0 + y1) / 2, lab, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white', rotation=270,
            clip_on=False, zorder=5)

trans_top = blended_transform_factory(ax.transData, ax.transAxes)
for name, i_s, i_e, col in reaction_groups:
    ax.add_patch(MplRect((i_s - 0.5, TOP_STRIP_Y), i_e - i_s + 1, TOP_STRIP_H,
                         transform=trans_top, facecolor=col, edgecolor='white',
                         lw=1.0, clip_on=False, zorder=4))
    ax.text((i_s + i_e) / 2, TOP_STRIP_Y + TOP_STRIP_H / 2, name, transform=trans_top,
            ha='center', va='center', fontsize=9, fontweight='bold',
            color='white', clip_on=False, zorder=5)

for s in ('top', 'right'): ax.spines[s].set_visible(False)
for s in ('bottom', 'left'): ax.spines[s].set_color('#2b2b2b'); ax.spines[s].set_linewidth(0.8)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label(r'Spearman $\rho(\theta_j,\, y_i)$', fontsize=11, labelpad=4)
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar.ax.tick_params(labelsize=9, length=3, width=0.5)
cbar.outline.set_linewidth(0.6)

fig.savefig('figures/sensitivity_oat_spearman_masked.pdf', dpi=300, bbox_inches='tight')
plt.show()
mpl.rcParams.update(mpl.rcParamsDefault)

